# 破解 CRC32：给你一个哈希值，把原始数据造回来

## 这一章要做的事

CRC32 是用来查数据完整性的哈希，但它不是密码学意义上的哈希。**输出一旦摆在眼前，你就能算出一个满足条件的输入来**。

- 把 CRC32 的计算，用 Z3 的 `BitVec` 一句一句照抄下来
- 以这段字符串开头、只有英文字母、CRC32 正好是这个值，这些都变成约束
- 剩下的交给 Z3，让它把符合条件的数据造出来

手头的工具就两样：`BitVec` 的位运算（`>>`, `&`, `^`, `~`），和 `ZeroExt`（把位宽撑开）。

## 什么是 CRC32

CRC32（Cyclic Redundancy Check 32-bit）是 32 位的校验和，专门用来发现数据里出的错。网络通信、文件完整性检查，到处都在用它。

可它**不是密码学意义上安全的哈希**。只要拿到一个输出，就能反推出产生这个输出的输入。为什么这东西不能拿来存密码，这一章你亲手试一遍就明白了。

## CRC32 的计算方法

CRC32 到底在算什么，看下面这段 C 代码就清楚了。

```c
static uint32_t crc32(uint32_t crc, unsigned char *buf, size_t len)
{
    int k;

    crc = ~crc;
    while (len--) {
        crc ^= *buf++;
        for (k = 0; k < 8; k++)
            crc = crc & 1 ? (crc >> 1) ^ 0xedb88320 : crc >> 1;
    }
    return ~crc;
}
```

拆开看，它只做四件事。

1. 先把初值按位取反（`crc = ~crc`）
2. 再把每个字节依次 XOR 进去
3. 然后重复 8 遍：右移 1 位，最低位是 1 就把多项式 `0xedb88320` XOR 上去
4. 最后再按位取反一次，返回

这套计算在 Python 里同样能一行行写出来。换成 Z3 的 `BitVec` 之后，**输入可以一直是变量，照样顺着算式往下走**，不用先把值填进去。

In [1]:
from z3 import *


def z3_crc32(data, crc=0):
    """返回 data（BitVec(8) 的列表）的 CRC32，形式为 Z3 的表达式"""
    crc = ~crc                      # 反转初始值
    for c in data:
        crc ^= ZeroExt(24, c)       # 把 8 位字节扩展到 32 位再 XOR
        for _ in range(8):
            crc = If(crc & 1 == BitVecVal(1, 32),
                     LShR(crc, 1) ^ 0xedb88320,
                     LShR(crc, 1))
    return ~crc                     # 最后反转

## 用到的 Z3 函数

| 函数与运算符 | 作用 |
|---|---|
| `ZeroExt(24, c)` | 把 8 位的值在高位补 0，撑成 32 位 |
| `BitVecVal(1, 32)` | 造一个 32 位常数 1（位宽是挂在类型上的） |
| `LShR(crc, 1)` | 无符号右移，空出来的高位补 0 |
| `If(条件, A, B)` | 条件为真就取 A，为假就取 B |
| `~crc` | 按位取反 |
| `^`, `&` | XOR 和 AND |

这里用 `LShR()` 而不用 `>>`，是因为 `BitVec` 分有符号和无符号两种。要跟 C 的 `unsigned` 走一样的路，右移就得用 `LShR()`，让空出来的高位补 0。

至于 `BitVecVal()`，它造出来的是“值固定的 `BitVec`”。旁边的 `simplify()` 负责化简式子：所有值都定下来的时候，结果就是一个数字，式子的样子也就没了。

## 先跟 `binascii.crc32()` 对一下

拿 Python 自带的 `binascii.crc32()` 比一比。同一个输入喂进去，两边出来的数一样，我们写的这段才算站得住。

In [2]:
from binascii import crc32

data = b'hello world!!'
print(crc32(data))

simplify(z3_crc32([BitVecVal(x, 8) for x in data]))

2909492980


2909492980

对上了。`data` 的每个字节都是用 `BitVecVal(x, 8)` 这样的常数喂进去的，所以 Z3 的式子一路化简到底，就是个数值。

## 从 CRC32 反推数据

正题从这里开始。下面这几条条件，交给 Z3 去找满足它们的数据。

- 长度是 `n` 字节
- 以 `head` 开头，以 `tail` 结尾
- 中间那些字符全是英文字母（`a`〜`z`, `A`〜`Z`）
- CRC32 等于 `crc_code`

把数据的每个字节都设成 `BitVec(f'b{i}', 8)` 这样的变量，“CRC32 等于 `crc_code`”本身就成了一条约束。

In [3]:
def is_between(b, start, end):
    """b 在 start 到 end 的字符编码范围内"""
    return And(b >= ord(start), b <= ord(end))


def starts_with(data, string):
    """data 以 string 开头"""
    return And([b == ord(ch) for b, ch in zip(data, string)])


def ends_with(data, string):
    """data 以 string 结尾"""
    return And([b == ord(ch) for b, ch in zip(data[::-1], string[::-1])])


def break_crc32(n, head, tail, crc_code):
    """找出满足条件的 n 字节数据"""
    data = [BitVec(f'b{i:02d}', 8) for i in range(n)]

    solver = Solver()
    solver.add(starts_with(data, head))
    solver.add(ends_with(data, tail))

    # 中间的字符只能是英文字母
    for b in data[len(head):-len(tail)]:
        solver.add(Or(is_between(b, 'a', 'z'), is_between(b, 'A', 'Z')))

    solver.add(z3_crc32(data) == crc_code)

    if solver.check() != sat:
        return None

    model = solver.model()
    return bytes([model[d].as_long() for d in data])


def test(n, head, tail, crc_code):
    res = break_crc32(n, head, tail, crc_code)
    if res is None:
        print('unsat（不存在满足条件的数据）')
    else:
        print(res, hex(crc32(res)))

## 执行示例

要找的是一段长度为 20 的字符串：`"hello "` 开头，`"!!"` 结尾，中间随便挑，只要求算出来的 CRC32 恰好等于 `0x12345678`。

In [4]:
test(20, 'hello ', '!!', 0x12345678)

b'hello PWvJSuTYDVsZ!!' 0x12345678


输出末尾那个 `0x...` 就是生成出来的字符串的 CRC32，跟你要的 `0x12345678` 对一对，应该相等。这里的 CRC 又用 `binascii.crc32()` 重算了一遍，所以 Z3 的答案到底对不对，一查便知。

## 太短就解不出来

字符数一少，能自由挑的字符也跟着少。这种情况下要是根本不存在满足条件的数据，Z3 只能回一个 `unsat`。

In [5]:
test(13, 'hello ', '!!', 0x00000000)

unsat（不存在满足条件的数据）


`head` 占 6 个字符，`tail` 占 2 个，能自由挑的中间部分就只剩 5 个字符。Z3 证明了：不管这 5 个字符怎么排，CRC32 都到不了 0。

那我们把长度加一个字节，再问它一次。

In [6]:
test(14, 'hello ', '!!', 0x00000000)

b'hello zXforl!!' 0x0


只不过是又多给了一个自由字符，数据就找到了。

- 能自由搜索的字符越多，满足条件的数据就越好找
- CRC32 的计算是线性的，条件原样写成约束，Z3 解得飞快
- 所以说，CRC32 拦得住检测篡改，拦不住防止篡改

## 回头看一眼

- CRC32 的计算，用 `BitVec` 的位运算能一行行照抄
- 8 位的字节靠 `ZeroExt(24, c)` 撑到 32 位
- 无符号右移用 `LShR()`，常数用 `BitVecVal()`
- 开头、结尾、字符种类、CRC 值这些条件写成约束，Z3 就能把数据造出来
- 自由字符太少，结果就是 `unsat`，也就是被证明了确实不存在